# Pairs Trading with Mean-Reverting Stochastic Processes
**MAC F244 project (Topic 17)**: Ornstein-Uhlenbeck spreads, ADF/Engle-Granger pair selection and optimal trading bands, on US and Indian stocks (2010-2026).

**How to use this notebook (temporary master copy)**
- Every section is tagged with its owner (M1-M5). Edit only your own sections, then tick your task in `Project_Tracker.csv`.
- The maths and the backtest live in `src/`; this notebook calls them and makes every figure and table in the report.
- Figures are saved to `figs/` as PNG and EPS with `savefig(fig, name)`, using the names `report/main.tex` expects.
- Cells marked **HEAVY** run the full walk-forward backtests (slow the first time, cached in `results/cache/` afterwards). They only run when `RUN_HEAVY = True`.
- Before submission, M1 runs *Restart & Run All* once and commits the outputs.

| Section | Content | Owner |
| --- | --- | --- |
| 1 | Setup | M1 |
| 2 | Data | M2 |
| 3 | OU model and simulation | M3 |
| 4 | E1 estimator accuracy, E2a ADF power, E2b test size | M3, M4, M2 |
| 5 | Pair selection on one window | M2 |
| 6 | Trading rules; Bertram bands and E3 | M4; M5 |
| 7 | Backtest engine checks, E4 synthetic pair | M1 |
| 8 | E5 headline backtest | M1, M4 |
| 9 | E6 transaction costs | M3 |
| 10 | E7 parameter sensitivity | M3 |
| 11 | E8 markets, sectors, sub-periods | M3, M2, M5 |
| 12 | E9 cointegration persistence | M2 |
| 13 | E10 Kalman hedge ratio (stretch) | M5 |
| 14 | Conclusions | M3 |

## 1. Setup  (Owner: M1)
Imports, the fixed seed (42) and the switch for the heavy backtests. All parameters live in `src/config.py`.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_prices, benchmark_returns, sectors
from src.ou import (simulate_ou, fit_ou, bertram_cycle_time, bertram_bands,
                    bertram_return_rate, first_passage_cycle_mc)
from src.cointegration import adf_test, engle_granger, engle_granger_both, naive_residual_adf
from src.selection import select_cointegrated, select_distance
from src.backtest import run_selection, simulate, pair_returns
from src.metrics import compute_metrics, metrics_table
from src.plots import savefig, COLORS

np.random.seed(C.SEED)
rng = np.random.default_rng(C.SEED)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

RUN_HEAVY = True  # True runs the full walk-forward backtests (sections 8-13)
STRATS = ['distance', 'zscore', 'sscore', 'bertram']

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import logging
logging.getLogger('matplotlib.backends.backend_ps').setLevel(logging.ERROR)

## 2. Data  (Owner: M2)
Daily adjusted closes from Yahoo Finance (`yfinance`), cached in `data/raw/` (see `data/raw/README.md` for retrieval date and flagged jumps). 56 US stocks in 7 sectors plus EWA/EWC, and 26 NSE stocks in 5 sectors. Pairs are formed only within a sector.

*M2: add your 5-line notes on Gatev et al. (2006) and Engle & Granger (1987) here (task T04).*

In [23]:
px_us = load_prices('us')
px_in = load_prices('india')
bench_us = benchmark_returns('us')
bench_in = benchmark_returns('india')

summary = pd.DataFrame({
    m: {'tickers': p.shape[1], 'first day': p.index[0].date(), 'last day': p.index[-1].date(),
        'trading days': len(p), 'missing %': round(100 * p.isna().mean().mean(), 2)}
    for m, p in {'US': px_us, 'India': px_in}.items()}).T
summary

,tickers,first day,last day,trading days,missing %
US,58,2010-01-04,2026-09-29,4210,0.000
India,26,2012-01-02,2026-09-29,3637,0.000


In [24]:
# Two classic pairs, prices normalised to 1 at the start
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('XOM', 'CVX')]):
    p = px_us[[a, b]].dropna()
    (p / p.iloc[0]).plot(ax=ax, lw=1)
    ax.set_title(f'{a} vs {b}'); ax.set_ylabel('Normalised price'); ax.set_xlabel('')
savefig(fig, 'fig_data_pairs')

## 3. The Ornstein-Uhlenbeck model  (Owner: M3)
The log-price spread $X_t = \ln P^A_t - \beta \ln P^B_t - \mu$ is modelled as
$$dX_t = \kappa(\theta - X_t)\,dt + \sigma\,dW_t, \qquad \kappa > 0.$$
Itô's lemma on $e^{\kappa t}X_t$ gives
$$X_t = \theta + (X_0-\theta)e^{-\kappa t} + \sigma\int_0^t e^{-\kappa(t-s)}\,dW_s,$$
so $X_t$ is Gaussian with mean $\theta + (X_0-\theta)e^{-\kappa t}$, variance $\sigma^2(1-e^{-2\kappa t})/(2\kappa)$, stationary law $N(\theta, \sigma^2/2\kappa)$ and half-life $\ln 2/\kappa$.

**Exact discretisation.** With step $\Delta = 1/252$:
$$X_{t+\Delta} = a + bX_t + \varepsilon_t,\quad b = e^{-\kappa\Delta},\quad a = \theta(1-b),\quad \mathrm{Var}(\varepsilon_t) = \sigma^2\frac{1-e^{-2\kappa\Delta}}{2\kappa}.$$
OLS of $X_{t+\Delta}$ on $X_t$ is the conditional MLE, and $\hat\kappa = -\ln\hat b/\Delta$, $\hat\theta = \hat a/(1-\hat b)$, $\sigma_{eq} = \hat s_\varepsilon/\sqrt{1-\hat b^2}$.

**Euler-Maruyama** uses $X_{t+\Delta} = X_t + \kappa(\theta - X_t)\Delta + \sigma\sqrt{\Delta}Z$, i.e. $b \approx 1-\kappa\Delta$. The noise is additive, so Milstein's correction term is zero and Milstein equals Euler-Maruyama.

*M3: check every line against report section 2 (task T05).*

In [18]:
# Exact vs Euler-Maruyama paths driven by the same shocks (same seed)
kappa, theta, sigma, n = 25.0, 0.0, 0.2, 252
x_exact = simulate_ou(kappa, theta, sigma, 0.08, n, method='exact', rng=np.random.default_rng(1))[0]
x_euler = simulate_ou(kappa, theta, sigma, 0.08, n, method='euler', rng=np.random.default_rng(1))[0]
sig_eq = sigma / np.sqrt(2 * kappa)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x_exact, lw=1, label='Exact scheme')
ax.plot(x_euler, lw=1, ls='--', label='Euler-Maruyama')
ax.axhspan(theta - sig_eq, theta + sig_eq, color='grey', alpha=0.15, label=r'$\theta \pm \sigma_{eq}$')
ax.set_xlabel('Trading day'); ax.set_ylabel('Spread'); ax.legend()
savefig(fig, 'fig_ou_paths')

# T11: the estimator recovers known parameters on a long exact path
rng_t11 = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
for k in [5, 25, 50]:
    x = simulate_ou(k, 0.0, 0.2, 'stationary', 252 * 40, rng=rng_t11)[0]
    p = fit_ou(x)
    print(f'kappa={k:>3}: kappa_hat={p.kappa:6.2f}, theta_hat={p.theta:+.4f}, sigma_hat={p.sigma:.4f}, half-life={p.half_life:5.1f} d')

kappa=  5: kappa_hat=  4.50, theta_hat=-0.0068, sigma_hat=0.2011, half-life= 38.8 d
kappa= 25: kappa_hat= 24.19, theta_hat=+0.0028, sigma_hat=0.2006, half-life=  7.2 d
kappa= 50: kappa_hat= 50.30, theta_hat=+0.0009, sigma_hat=0.2004, half-life=  3.5 d


## 4. E1 and E2: can we estimate and detect mean reversion?  (Owners: M3 E1, M4 E2a, M2 E2b)
**E1.** Monte Carlo of $\hat\kappa$ for windows of 63-504 days. Yu (2012) shows $\hat\kappa$ is biased upward in short samples, which makes half-lives look shorter than they are. We also compare the exact estimator with the Euler estimator $(1-\hat b)/\Delta$.

**E2.** The ADF regression $\Delta X_t = \alpha + \gamma X_{t-1} + \sum_i\delta_i\Delta X_{t-i} + \varepsilon_t$ has $\gamma = e^{-\kappa\Delta}-1$, so $H_0:\gamma=0$ is $H_0:\kappa=0$. We measure (a) the power of ADF across half-life and window length, which justifies the 12-month formation window, and (b) the size of the residual test on two independent random walks: plain ADF critical values vs Engle-Granger (MacKinnon 2010) critical values (spurious regression, Granger & Newbold 1974).

In [19]:
# E1: bias and RMSE of kappa_hat
rng_e1 = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
kappas, lengths, n_paths = [5, 10, 25, 50], [63, 126, 252, 504], 1000
rows = []
for k in kappas:
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=n_paths, rng=rng_e1)
        k_exact = np.array([fit_ou(p).kappa for p in x])
        k_euler = np.array([fit_ou(p, estimator='euler').kappa for p in x])
        rows.append(dict(kappa=k, days=T, valid_share=np.isfinite(k_exact).mean(),
                         rel_bias_exact=np.nanmean(k_exact) / k - 1,
                         rel_rmse_exact=np.sqrt(np.nanmean((k_exact - k) ** 2)) / k,
                         rel_bias_euler=np.nanmean(k_euler) / k - 1))
e1 = pd.DataFrame(rows)
e1.to_csv('results/e1_ou_estimation.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 3.6))
for k in kappas:
    d = e1[e1.kappa == k]
    ax.plot(d.days, 100 * d.rel_bias_exact, marker='o', label=f'kappa = {k} / yr')
ax.set_xscale('log'); ax.set_xticks(lengths); ax.set_xticklabels(lengths); ax.minorticks_off()
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('Relative bias of kappa_hat (%)'); ax.legend()
savefig(fig, 'fig_e1_ou_estimation')
e1

,kappa,days,valid_share,rel_bias_exact,rel_rmse_exact,rel_bias_euler
0,5,63,0.984,4.331,5.793,3.932
1,5,126,0.987,1.962,2.805,1.840
2,5,252,0.998,0.960,1.430,0.912
3,5,504,1.000,0.458,0.767,0.433
4,10,63,0.994,2.146,3.050,1.878
5,10,126,1.000,0.939,1.437,0.845
6,10,252,1.000,0.460,0.799,0.411
7,10,504,1.000,0.206,0.439,0.174
8,25,63,1.000,0.742,1.225,0.562
9,25,126,1.000,0.372,0.694,0.268


In [20]:
# E2a: ADF power over half-life x window length (rejection rate at 5%)
half_lives, lengths, reps = [5, 10, 20, 40, 80], [63, 126, 252, 504], 300
rng_e2 = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
power = pd.DataFrame(index=half_lives, columns=lengths, dtype=float)
for hl in half_lives:
    k = np.log(2) / (hl * C.DT)
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=reps, rng=rng_e2)
        power.loc[hl, T] = np.mean([adf_test(p, maxlag=5)[1] < 0.05 for p in x])
power.to_csv('results/e2_adf_power.csv')

fig, ax = plt.subplots(figsize=(5.5, 3.6))
im = ax.imshow(power.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(lengths)), lengths); ax.set_yticks(range(len(half_lives)), half_lives)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('True half-life (days)')
for i in range(len(half_lives)):
    for j in range(len(lengths)):
        ax.text(j, i, f'{power.values[i, j]:.2f}', ha='center', va='center', color='k' if power.values[i, j] > 0.6 else 'w', fontsize=8)
fig.colorbar(im, label='ADF rejection rate')
ax.grid(False)
savefig(fig, 'fig_e2_adf_power')
power

,63,126,252,504
5,0.297,0.717,0.990,1.000
10,0.163,0.253,0.737,0.997
20,0.133,0.127,0.303,0.730
40,0.087,0.080,0.133,0.257
80,0.097,0.057,0.090,0.100


In [37]:
# E2b: size of the residual test when the truth is 'not cointegrated'
reps, n = 1000, 252
naive, eg = [], []
for _ in range(reps):
    y, x = np.cumsum(rng.standard_normal((2, n)) * 0.01, axis=1)
    naive.append(naive_residual_adf(y, x)[1] < 0.05)
    eg.append(engle_granger(y, x).pvalue < 0.05)
e2_size = pd.Series({'Plain ADF critical values on residuals': np.mean(naive),
                     'Engle-Granger (MacKinnon) critical values': np.mean(eg)},
                    name='Rejection rate of a nominal 5% test')
e2_size.to_csv('results/e2_size.csv')
e2_size

Plain ADF critical values on residuals      0.169
Engle-Granger (MacKinnon) critical values   0.062
Name: Rejection rate of a nominal 5% test, dtype: float64

*M3 (E1), M4 (E2a), M2 (E2b): interpret your experiment here in 2 sentences each (bias size at 252 days, which half-lives a 12-month window can detect, how badly the naive test over-rejects).*

**E1 (M3).** $\hat\kappa$ is biased upward in every case, and the bias falls roughly in proportion to window length: on our 252-day formation window it overstates $\kappa$ by 18% for a 7-day half-life ($\kappa=25$) and by 96% for a 35-day half-life ($\kappa=5$); the absolute bias is nearly constant at about 4.5 per year, as Yu (2012) predicts for an estimated mean. Fitted half-lives are therefore too short. The Euler estimator looks less biased only because its own downward error of about $\kappa\Delta/2$ partly cancels the upward bias, so we keep the exact estimator.

**E2a (M4).** With the 12-month formation window ADF rejects a unit root 99% of the time for a 5-day half-life and 74% for 10 days, but only 30% for 20 days and 9-13% for 40-80 days, barely above the 5% size. A 252-day window therefore detects only fast mean reversion (half-lives up to about 10 days); in our US selections no pair has a fitted half-life above 21 days (E5b), so the 63-day upper filter never binds: slow pairs simply never pass, and the upward bias in $\hat\kappa$ from E1 makes fitted half-lives look shorter still.

## 5. Pair selection on one formation window  (Owner: M2)
Engle-Granger in both directions (keep a pair only if both p-values are below 5%), positive hedge ratio, half-life 2-63 trading days, at least 12 mean crossings a year (Sarmento & Horta 2020), ranked by the EG statistic, top 20. Benjamini-Hochberg (q = 0.10) is reported as a diagnostic. These numbers go into the proposal's *progress so far* paragraph (task T13).

In [ ]:
form = np.log(px_us.iloc[-C.FORMATION_DAYS:]).dropna(axis=1)
print(f'Formation window: {form.index[0].date()} to {form.index[-1].date()}')
chosen, diag, tested = select_cointegrated(form, sectors('us'))
print(diag)
top = pd.DataFrame([dict(pair=f'{s.a}/{s.b}', sector=s.sector, beta=s.beta, eg_stat=s.eg_stat,
                         eg_p=s.eg_pvalue, half_life_days=s.half_life, crossings_per_year=s.crossings)
                    for s in chosen])
top.head(10)

## 6. Trading rules and Bertram's optimal bands  (Owners: M4 rules, M5 Bertram and E3)
- **S0 distance** (Gatev et al. 2006): open when normalised prices diverge by 2 formation standard deviations, close at the crossing.
- **S1 z-score**: open at $|z|>2$, close at $|z|<0.5$.
- **S2 s-score** (Avellaneda & Lee 2010): $s = (X-\hat\theta)/\sigma_{eq}$; open long $s<-1.25$, open short $s>1.25$, close long $s>-0.50$, close short $s<0.75$.
- **S3 Bertram (2010)**: for the centred OU with entry $a$, exit $m$ and round-trip cost $c$,
$$\mathbb{E}[T] = \frac{\pi}{\kappa}\left[\mathrm{erfi}\left(\frac{m\sqrt\kappa}{\sigma}\right) - \mathrm{erfi}\left(\frac{a\sqrt\kappa}{\sigma}\right)\right],\qquad \mu(a,m) = \frac{m-a-c}{\mathbb{E}[T]},$$
from the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$. With $a=-m$ we maximise $\mu$ numerically.

**Notes on Avellaneda & Lee (2010) and Bertram (2010) (M4, T06)**
1. *Avellaneda & Lee* trade the residual of each US stock's returns after regressing on sector ETFs or PCA factors; the cumulated residual is fitted as an OU process on a 60-day window.
2. Their trading signal is the s-score $s=(X-m)/\sigma_{eq}$; thresholds 1.25 to open and 0.50 / 0.75 to close were chosen once, not optimised per stock, and only stocks with fast mean reversion ($\kappa > 252/30$, reversion time under half the window) are traded.
3. They report Sharpe ratios above 1 for 1997-2007 that fall after 2002 and around the August 2007 quant crisis, i.e. the edge decays as it becomes crowded. We use their thresholds unchanged on pairs (S2).
4. *Bertram* solves for the expected round-trip time of an OU process between an entry band $a$ and exit band $m$ via first-passage times (the erfi formula above), and the variance of the return per trade.
5. Maximising expected return per unit time gives symmetric bands $a=-m$ whose width grows with the transaction cost; maximising the Sharpe ratio instead gives wider bands. We implement the return-maximising version (S3).

**Notes on Bertram (2010) and the Kalman filter (M5, T49; full viva note in notes/M5_Bertram_Kalman_viva_note.md)**
1. Bertram models the centred spread as an OU process and treats a trade as one cycle: enter at $a$, exit at $m$, wait to return to $a$.
2. The expected cycle time comes from first-passage times, which solve the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$; integrating twice gives the erfi formula above.
3. Dividing the profit per cycle, $m - a - c$, by $\mathbb{E}[T]$ gives the return per unit time; with symmetric bands its maximiser $m^*$ depends only on $c/\sigma_{eq}$, not on $\kappa$, and shrinks to 0 when trading is free.
4. The Kalman filter (Elliott et al. 2005; Chan 2013) treats $(\mu_t, \beta_t)$ as a random walk and updates them each day from the forecast error, so the hedge ratio can drift.
5. Its noise settings (state variance $\delta$, observation variance) decide how fast $\beta_t$ adapts: too fast and the filter absorbs the very spread we want to trade (E10).

In [ ]:
# T43 (M4): check the z-score (S1) and s-score (S2) rules on a toy spread worked out by hand
from src.strategies import positions
from src.selection import PairSpec

toy = np.array([0, 1.0, 2.1, 1.5, 0.6, 0.4, -0.3, -2.2, -1.0, -0.6, -0.4, 0.2, 2.5, -2.5, 0.0])
unit = PairSpec('A', 'B', 'toy', 'coint', beta=1.0, mu=0.0, sd=1.0, theta=0.0, sigma_eq=1.0)
s1, s2 = positions('zscore', toy, unit), positions('sscore', toy, unit)
assert (s1 == [0, 0, -1, -1, -1, 0, 0, 1, 1, 1, 0, 0, -1, 1, 0]).all()   # open |z|>2, close inside 0.5
assert (s2 == [0, 0, -1, -1, 0, 0, 0, 1, 1, 1, 0, 0, -1, 1, 0]).all()   # 1.25 / -0.50 / 0.75
assert s2[4] == 0 and s1[4] == -1        # at 0.6 the S2 short closes (< 0.75) but S1 stays open (> 0.5)
# S2 centres on the OU mean theta and scales by sigma_eq; S1 does not
shifted = PairSpec('A', 'B', 'toy', 'coint', 1.0, 0.0, 1.0, theta=0.5, sigma_eq=0.5)
assert (positions('sscore', toy, shifted) == positions('sscore', (toy - 0.5) / 0.5, unit)).all()
# Stop-loss at |s| = 4 closes the trade and blocks re-entry until s is back inside the entry band
assert (positions('sscore', np.array([0, 2.0, 4.5, 3.0, 1.0, 2.0]), unit, stop=4.0) == [0, -1, 0, 0, 0, -1]).all()

# E5c time stop (k = 3 half-lives; half-life 1 day here, so trades close after 3 days held)
fast = PairSpec('A', 'B', 'toy', 'coint', 1.0, 0.0, 1.0, theta=0.0, sigma_eq=1.0, half_life=1.0)
stuck = np.array([0, 2.0, 1.8, 1.7, 1.6, 1.5, 1.4, 0.0, 2.0])
assert (positions('sscore', stuck, fast) == [0, -1, -1, -1, -1, -1, -1, 0, -1]).all()          # headline: waits for 0.75
assert (positions('sscore', stuck, fast, params={'time_stop': 3}) == [0, -1, -1, -1, 0, 0, 0, 0, -1]).all()  # closed, blocked until back in band
flip = np.array([0, 2.0, 1.8, -2.0, -1.8, -1.7, -1.6])
assert (positions('sscore', flip, fast, params={'time_stop': 3}) == [0, -1, -1, 1, 1, 1, 0]).all()     # clock restarts on a flip
assert (positions('zscore', stuck, fast, params={'time_stop': 3}) == [0, 0, 0, 0, 0, 0, 0, 0, 0]).all()  # |z| never > 2 here
assert (positions('zscore', toy, unit) == positions('zscore', toy, unit, params={'time_stop': None})).all()

# On simulated OU spreads (half-life 10 days) both rules earn money before costs out of sample
pnl = {'zscore': [], 'sscore': []}
for x in simulate_ou(np.log(2) / (10 * C.DT), 0, 0.2, 'stationary', 504, n_paths=200, rng=np.random.default_rng(C.SEED)):
    ou = fit_ou(x[:252])                                      # fit on the first year, trade the second
    spec = PairSpec('A', 'B', 'sim', 'coint', 1.0, 0.0, x[:252].std(), theta=ou.theta, sigma_eq=ou.sigma_eq)
    for r in pnl:
        p = positions(r, x[252:], spec)
        pnl[r].append((p[:-1] * np.diff(x[252:])).sum())
pd.DataFrame({r: {'mean spread P&L': np.mean(v), 'share of paths > 0': np.mean(np.array(v) > 0)} for r, v in pnl.items()}).T

In [ ]:
# Proposal figure (T12): spread as an s-score with the S2 bands, latest formation year.
# Shading shows where the s-score rule would hold the spread (red short, green long).
from src.strategies import positions
from src.selection import PairSpec
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('EWA', 'EWC')]):
    lp = np.log(px_us[[a, b]].iloc[-C.FORMATION_DAYS:].dropna())
    best, p_both = engle_granger_both(lp[a].values, lp[b].values, a, b)
    ou = fit_ou(best.resid)
    s = (best.resid - ou.theta) / ou.sigma_eq
    spec = PairSpec(best.y, best.x, '', 'coint', best.beta, best.mu, best.resid.std(),
                    theta=ou.theta, sigma_eq=ou.sigma_eq)
    pos = positions('sscore', best.resid, spec)
    ax.fill_between(lp.index, -3, 3, where=pos > 0, color='tab:green', alpha=0.12, lw=0)
    ax.fill_between(lp.index, -3, 3, where=pos < 0, color='tab:red', alpha=0.12, lw=0)
    ax.plot(lp.index, s, lw=1, color='k')
    ax.axhline(1.25, color='tab:red', lw=0.8, label='open short / long (±1.25)')
    ax.axhline(-1.25, color='tab:green', lw=0.8)
    ax.axhline(0.75, color='tab:red', ls=':', lw=0.8, label='close short (0.75)')
    ax.axhline(-0.50, color='tab:green', ls=':', lw=0.8, label='close long (-0.50)')
    sign = '-' if best.beta >= 0 else '+'
    ax.set_title(f'{best.y} {sign} {abs(best.beta):.2f} {best.x}: EG p = {p_both:.2f}, '
                 f'half-life {ou.half_life:.0f} d', fontsize=10)
    ax.set_ylim(-3, 3); ax.tick_params(axis='x', labelrotation=30)
axes[0].set_ylabel('s-score')
axes[0].legend(fontsize=7, loc='lower left')
savefig(fig, 'fig_proposal_spread')

In [ ]:
# T50 (M5): check bertram_cycle_time() and bertram_bands() against the theory
from scipy.integrate import quad
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)

# (1) The erfi formula is the solution of the backward equation (1/2) sigma^2 u'' - kappa x u' = -1.
#     Up-crossing a -> m plus down-crossing m -> a reduces to (2 / sigma^2) * int_a^m s(y) dy * int s(z)^-1 dz,
#     with scale density s(y) = exp(kappa y^2 / sigma^2). Integrate that numerically and compare.
gauss = quad(lambda z: np.exp(-kappa * z * z / sigma ** 2), -np.inf, np.inf)[0]
rows = []
for m in [0.25, 0.5, 1.0, 2.0]:
    mm = m * sig_eq
    T_quad = 2 / sigma ** 2 * quad(lambda y: np.exp(kappa * y * y / sigma ** 2), -mm, mm)[0] * gauss
    T_erfi = bertram_cycle_time(-mm, mm, kappa, sigma)
    rows.append({'m / sigma_eq': m, 'quadrature (days)': 252 * T_quad, 'erfi formula (days)': 252 * T_erfi})
    assert abs(T_quad / T_erfi - 1) < 1e-8

# (2) The optimiser finds the maximum of mu(-m, m) = (2m - c) / E[T]: compare with a fine grid
for c in [0.2, 0.5, 1.0]:
    cc = c * sig_eq
    grid = np.linspace(cc / 2 * 1.0001, 4 * sig_eq, 20000)
    best = grid[np.argmax(bertram_return_rate(grid, kappa, sigma, cc))]
    assert abs(bertram_bands(kappa, sigma, cc) - best) < 1e-3 * sig_eq

# (3) Shape: m* rises with the cost and tends to 0 when trading is free (Bertram 2010)
m_star = [bertram_bands(kappa, sigma, c * sig_eq) / sig_eq for c in [0, 0.1, 0.5, 1.0, 2.0]]
assert m_star[0] < 0.01 and np.all(np.diff(m_star) > 0)
print('m* / sigma_eq at cost 0, 0.1, 0.5, 1, 2 sigma_eq:', np.round(m_star, 3))
pd.DataFrame(rows)

In [ ]:
# E3 (M5): Bertram's E[T] against simulated cycles, and m* against cost
# Each simulation covers the same 300 years of OU time, so a finer monitoring step does not mean fewer cycles.
# The cycle -m -> m -> -m is detected only at monitoring times; coarse monitoring misses crossings.
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)
rng_e3 = np.random.default_rng(C.SEED)       # own seed: same numbers whatever ran before
years = 300
m_grid = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0])
steps_per_day = {'1 day': 1, '1/10 day': 10, '1/200 day': 200}
rows = []
for m in m_grid:
    row = {'m / sigma_eq': m, 'formula (days)': 252 * bertram_cycle_time(-m * sig_eq, m * sig_eq, kappa, sigma)}
    for name, k in steps_per_day.items():
        cyc = 252 * first_passage_cycle_mc(kappa, sigma, m * sig_eq, n_steps=years * 252 * k, dt=C.DT / k, rng=rng_e3)
        row[f'MC {name}'] = cyc.mean()
        row[f'SE {name}'] = cyc.std(ddof=1) / np.sqrt(len(cyc))
    rows.append(row)
e3 = pd.DataFrame(rows)
e3.to_csv('results/e3_bertram_check.csv', index=False)

costs = np.linspace(0, 1.0, 21) * sig_eq
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(e3['m / sigma_eq'], e3['formula (days)'], color='k', label='Bertram formula')
for name, mk in [('1/200 day', 'o'), ('1 day', 's')]:
    axes[0].errorbar(e3['m / sigma_eq'], e3[f'MC {name}'], yerr=2 * e3[f'SE {name}'], fmt=mk, ms=4,
                     capsize=2, label=f'Monte Carlo, checked every {name}')
axes[0].set_xlabel('Band half-width m / sigma_eq'); axes[0].set_ylabel('Expected cycle (days)'); axes[0].legend(fontsize=8)
# m* / sigma_eq depends only on cost / sigma_eq: kappa scales E[T] but not the argmax. Speed sets the return instead.
m_star = np.array([bertram_bands(kappa, sigma, c) for c in costs])
axes[1].plot(costs / sig_eq, m_star / sig_eq, color='k', label='Optimal m* (any kappa)')
axes[1].set_xlabel('Round-trip cost / sigma_eq'); axes[1].set_ylabel('Optimal m* / sigma_eq')
ax_r = axes[1].twinx()
for k, ls in [(5, ':'), (20, '--'), (50, '-.')]:
    sigma_k = sig_eq * np.sqrt(2 * k)   # same sigma_eq, different speed
    assert np.allclose([bertram_bands(k, sigma_k, c) for c in costs], m_star, rtol=1e-4, atol=1e-6 * sig_eq)
    ax_r.plot(costs / sig_eq, bertram_return_rate(m_star, k, sigma_k, costs) / sig_eq, ls=ls, color='tab:blue',
              label=f'return per year at m*, kappa = {k}')
ax_r.set_ylabel('Return per year / sigma_eq', color='tab:blue'); ax_r.grid(False)
h1, l1 = axes[1].get_legend_handles_labels(); h2, l2 = ax_r.get_legend_handles_labels()
axes[1].legend(h1 + h2, l1 + l2, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
savefig(fig, 'fig_e3_bertram')
e3.round(2)

**E3 result (M5).** (results/e3_bertram_check.csv, figs/fig_e3_bertram)

- **The formula and the code are right (T50).** The erfi formula equals a direct numerical integration of the backward-equation solution to $10^{-8}$, the optimiser matches a 20,000-point grid, and $m^*$ rises with cost and tends to 0 when trading is free.
- **The simulation converges to the formula from above.** At the narrowest band ($0.25\sigma_{eq}$) the simulated cycle is 99% too long when the spread is checked once a day, 28% at 1/10 day and 5% at 1/200 day; wider bands are within about 2 standard errors at 1/200 day. The gap is discrete-monitoring bias (a path can cross a band and come back between two checks), not an error in the formula. The first draft compared the formula with a fixed number of 1/50-day steps, which mixed this bias with too few cycles.
- **Consequence for trading.** A rule that checks the spread once a day completes cycles 35-99% more slowly than the continuous-time formula assumes, so Bertram's return per unit time is optimistic for daily trading, most of all for narrow bands. This is part of the gap M1 measures in E4 (daily monitoring and the one-day lag cost 3.3 of the 24.2 points).
- **Right panel.** $m^*/\sigma_{eq}$ depends only on the cost relative to $\sigma_{eq}$: the three speeds give the same band. Speed changes the return instead, which at the optimum is proportional to $\kappa$ (the rate is $\kappa$ times a function of $m/\sigma_{eq}$ and $c/\sigma_{eq}$).

## 7. Backtest engine checks and E4 synthetic pair  (Owner: M1)
**Timing rules (no look-ahead).** Pairs, tickers and every parameter come from the 252-day formation window only. The signal computed at the close of day $t$ is executed at the close of $t+1$ (the one-day wait of Gatev et al. 2006), so day $t$'s return is earned by the position decided at the close of $t-2$. Open positions are closed on the last day of the 126-day trading window.

**P&L.** One unit of spread is \$1 long $A$ and \$$\beta$ short $B$. Since $r^A-\beta r^B \approx d\ln P^A-\beta\,d\ln P^B = dX$, the position earns the change of the OU spread. Daily return on gross capital: $\text{pos}_{t-1}(r^A_t-\beta r^B_t)/(1+|\beta|)$, minus the cost per leg on every unit traded (a flip long$\to$short is two units).

**Checks.** `tests/` holds 20 tests with hand-computable answers. The decisive one scrambles every price after a date (and deletes some) and checks that no earlier return changes.

In [ ]:
# T14, T22-T24: unit tests of the engine and the metrics, then the look-ahead check by hand
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True).stdout.strip().splitlines()[-1])

desired = np.zeros(30); desired[10:20] = 1          # signal on at the close of day 10
ra = np.full(30, 0.01); rb = np.zeros(30)
net, held = pair_returns(desired, 1.0, ra, rb, cost=0.0)
assert held[10] == 0 and held[11] == 1, 'execution lag broken'
assert net[11] == 0 and net[12] > 0, 'return earned before the position was held'
print('Look-ahead check passed: signal at close 10, bought at close 11, first return on day 12')

**Integration check (T25).** One real pair through the whole pipeline on one window: selection on 2018, trading in the first half of 2019.

In [ ]:
from src.backtest import make_windows
w = make_windows(len(px_us))[16]
chosen, diag, _ = select_cointegrated(np.log(px_us.iloc[w.f0:w.t0]), sectors('us'))
spec = chosen[0]
print(f'Formation {px_us.index[w.f0].date()} to {px_us.index[w.t0 - 1].date()}, trading {px_us.index[w.t0].date()} to {px_us.index[w.t1 - 1].date()}')
print(f'{diag["n_eg"]}/{diag["n_tested"]} pairs pass Engle-Granger, {diag["n_filtered"]} pass all filters')
print(f'Top pair {spec.a}/{spec.b} ({spec.sector}): beta = {spec.beta:.3f}, EG p = {spec.eg_pvalue:.4f}, '
      f'kappa = {spec.kappa:.1f}/yr, half-life = {spec.half_life:.1f} d, sigma_eq = {spec.sigma_eq:.4f}')
one = [dict(window=w, coint=[spec], distance=[], diag=diag)]
res_one = simulate('sscore', px_us, one, C.COST_BPS['us'], k=1)

lp = np.log(px_us[[spec.a, spec.b]].iloc[w.f0:w.t1])
s = (lp[spec.a] - spec.beta * lp[spec.b] - spec.mu - spec.theta) / spec.sigma_eq
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(s.index, s.values, lw=0.9, color=COLORS['sscore'])
ax.axvspan(px_us.index[w.t0], px_us.index[w.t1 - 1], color='grey', alpha=0.12)
ax.text(px_us.index[w.t0 + 3], -3.0, 'trading window: parameters frozen', fontsize=8)
ax.set_ylim(-3.3, 2.7)
for lvl in (1.25, -1.25): ax.axhline(lvl, color='grey', lw=0.8)
for lvl in (0.75, -0.5): ax.axhline(lvl, color='grey', lw=0.8, ls=':')
for _, t in res_one.trades.iterrows():
    ax.axvspan(t.open, t.close, ymin=0.92, ymax=1.0, color='k' if t.side > 0 else 'grey')
ax.set_ylabel('s-score'); ax.set_title(f'{spec.a} - {spec.beta:.2f} {spec.b}: bars on top mark trades (black long, grey short)')
savefig(fig, 'fig_m1_integration')
print(compute_metrics(res_one.committed, trades=res_one.trades)['Total return %'], '% on the pair over the window')
res_one.trades

**E4 (T27): does the pipeline earn what the OU theory predicts?** A synthetic pair with a known OU spread ($\kappa = 25$/yr, half-life 7 days, $\sigma_{eq} = 0.03$, $\beta = 1.2$, 5 bp per leg) is pushed through the pipeline 200 times. For this pair Bertram's formula predicts the return of the optimal bands exactly, so we add one real-world friction at a time and measure what each costs:

| Level | What changes |
| --- | --- |
| L0 | Bertram's formula: continuous monitoring, true parameters |
| L1 | spread seen once a day, traded at the same close, true parameters |
| L2 | + one-day execution lag |
| L3 | + 6-month trading windows (start flat, forced close) |
| L4a | + OU parameters $\kappa,\theta,\sigma$ estimated on the formation year (true $\beta$) |
| L4 | + $\beta$ and $\mu$ estimated too |
| L5 | + pair traded only when it passes Engle-Granger and the OU filters (the full pipeline) |

In [ ]:
# E4 ladder: 200 synthetic pairs, about 90 s; cached in results/
import json, os
from src.synthetic import run_ladder
if os.path.exists('results/e4_ladder.csv') and os.path.exists('results/e4_theory.json'):
    e4 = pd.read_csv('results/e4_ladder.csv'); e4_info = json.load(open('results/e4_theory.json'))
else:
    e4, e4_info = run_ladder(reps=200)
    e4.to_csv('results/e4_ladder.csv', index=False)
    json.dump({k: float(v) for k, v in e4_info.items()}, open('results/e4_theory.json', 'w'), indent=1)

levels = ['L1', 'L2', 'L3', 'L4a', 'L4', 'L5']
names = {'L0': 'L0 Bertram formula', 'L1': 'L1 daily monitoring', 'L2': 'L2 + one-day lag', 'L3': 'L3 + 6-month windows',
         'L4a': 'L4a + estimated OU params', 'L4': 'L4 + estimated beta', 'L5': 'L5 + EG selection (pipeline)'}
n = len(e4)
table_e4 = pd.DataFrame({'Ann. return %': [100 * e4_info['theory']] + [100 * e4[l].mean() for l in levels],
                         'SE %': [np.nan] + [100 * e4[l].std() / np.sqrt(n) for l in levels]},
                        index=[names[l] for l in ['L0'] + levels])
steps = [e4['L1'] - e4_info['theory']] + [e4[b] - e4[a] for a, b in zip(levels, levels[1:])]
table_e4['Step %'] = [np.nan] + [100 * d.mean() for d in steps]
table_e4['Step SE %'] = [np.nan, np.nan] + [100 * d.std() / np.sqrt(n) for d in steps[1:]]   # paired SE
table_e4['Share of theory %'] = 100 * table_e4['Ann. return %'] / table_e4['Ann. return %'].iloc[0]
table_e4.to_csv('results/e4_table.csv')
print(f"Fine-grid simulation of L0 (50 steps a day, 200 years): {100 * e4_info['fine_grid']:.1f}% vs formula {100 * e4_info['theory']:.1f}%")
print(f"Pair selected in {100 * e4.selected_share.mean():.0f}% of windows; median kappa_hat / kappa = {e4.kappa_hat_over_true.mean():.2f}; "
      f"median |beta_hat - beta| = {e4.beta_abs_err.mean():.3f}; m*/sigma_eq true {e4_info['m_star']:.3f}, fitted {e4.m_hat_in_sig_eq.mean():.3f}")
print(f"s-score on the same pipeline (L5): {100 * e4.sscore_L5.mean():.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.4))
y = np.arange(len(table_e4))[::-1]
ax.barh(y, table_e4['Ann. return %'], height=0.6, color=COLORS['bertram'], xerr=2 * table_e4['SE %'].fillna(0),
        error_kw=dict(lw=1, capsize=2, ecolor='k'))
ax.axvline(100 * e4_info['theory'], color='k', lw=0.8, ls='--')
for yi, v, se in zip(y, table_e4['Ann. return %'], table_e4['SE %'].fillna(0)):
    ax.text(v + 2 * se + 0.4, yi, f'{v:.1f}%', va='center', fontsize=8, bbox=dict(fc='white', ec='none', pad=0.5))
ax.set_yticks(y, table_e4.index); ax.set_xlabel('Annual return on gross capital (%), mean of 200 pairs, ±2 SE')
ax.set_xlim(0, 100 * e4_info['theory'] * 1.15); ax.grid(axis='y', visible=False)
savefig(fig, 'fig_e4_synthetic')
table_e4.round(2)

**E4 result.** The fine-grid simulation reproduces Bertram's formula (24.3% vs 24.2% a year), so the formula and our P&L bookkeeping agree. The full pipeline earns 12.7% (SE 0.2%), about half the theoretical return, and every step of the gap has a cause:

* **Daily monitoring (−1.1 pp).** The spread can cross a band and come back within a day, so some cycles are missed. Each daily step moves $0.42\,\sigma_{eq}$, almost as much as the band half-width $0.49\,\sigma_{eq}$.
* **One-day lag (−2.2 pp).** By the time we trade, the spread has already reverted by about $1-e^{-\kappa\Delta}\approx 9\%$ of its distance from the mean. This is the price of an honest execution rule.
* **Window resets (−0.5 pp).** Each window starts flat and ends with a forced close.
* **Estimated OU parameters (−1.0 pp).** $\hat\kappa$ is biased upward by about 24% at 252 days (Yu 2012, our E1), but the optimal band in units of $\sigma_{eq}$ barely moves ($0.50$ vs $0.49$), so the loss is small.
* **Estimated hedge ratio (−2.3 pp).** A median error of only $|\hat\beta-\beta| = 0.064$ adds $(\beta-\hat\beta)\ln P^B$ to the traded spread: a random walk of typical size $0.064\times0.015\times\sqrt{126}\approx 0.36\,\sigma_{eq}$ over a trading window, drifting against bands only $0.49\,\sigma_{eq}$ wide. Hedge-ratio error costs more than all OU-parameter errors together (motivates the Kalman hedge, E10).
* **Selection (−4.5 pp).** Engle-Granger at 5% in both directions recognises this truly cointegrated pair in only 74% of formation windows (the power limit measured in E2), and the capital sits idle otherwise: $0.74 \times 17.2\% = 12.7\%$, exactly L5.

The s-score rule on the same pipeline earns 9.7%, below Bertram's 12.7%: on a true OU pair the bands built for maximum return per unit time do win. A single 12-year run is noisy (standard deviation 2.9% across pairs; the first seed we tried gave 8.6%), which is why E4 averages 200 pairs.

## 8. E5 headline backtest, US  (Owners: M1 engine, M4 strategies)  HEAVY
Rolling 12-month formation and 6-month trading windows, K = 20 pairs, 10 bp per leg. Design choices are frozen on 2010-2021; 2022-2026 is the untouched test block.

In [25]:
if RUN_HEAVY:
    sel_us = run_selection('us', px_us, sectors('us'))     # slow once, then cached
    res_us = {s: simulate(s, px_us, sel_us, C.COST_BPS['us'], label=s) for s in STRATS}
    table1 = metrics_table(res_us.values(), bench_us).rename(index=C.STRATEGY_LABELS)
    table1.to_csv('results/table1_us.csv')
    split = pd.DataFrame({
        (C.STRATEGY_LABELS[s], blk): compute_metrics(r.committed[sl])
        for s, r in res_us.items()
        for blk, sl in [('2011-2021', slice(None, '2021-12-31')), ('2022-2026 test', slice(C.TEST_START, None))]
    }).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    split.to_csv('results/table1_split.csv')
    display(table1.round(2)); display(split.round(2))

  us window  0 2011-01-03:   8/196 pass EG,  8 pass filters
  us window  1 2011-07-05:  12/196 pass EG, 12 pass filters
  us window  2 2012-01-03:  11/196 pass EG, 11 pass filters
  us window  3 2012-07-03:   9/196 pass EG,  9 pass filters
  us window  4 2013-01-04:   5/196 pass EG,  5 pass filters
  us window  5 2013-07-08:  12/196 pass EG, 11 pass filters
  us window  6 2014-01-06:  10/196 pass EG, 10 pass filters
  us window  7 2014-07-08:  12/196 pass EG, 12 pass filters
  us window  8 2015-01-06:  16/196 pass EG, 16 pass filters
  us window  9 2015-07-08:  11/196 pass EG, 11 pass filters
  us window 10 2016-01-06:  10/196 pass EG,  7 pass filters
  us window 11 2016-07-07:   9/196 pass EG,  9 pass filters
  us window 12 2017-01-05:  12/196 pass EG, 12 pass filters
  us window 13 2017-07-07:  10/196 pass EG,  9 pass filters
  us window 14 2018-01-05:   5/196 pass EG,  5 pass filters
  us window 15 2018-07-09:   9/196 pass EG,  9 pass filters
  us window 16 2019-01-08:   6/196 pass 

,Ann. return %,Ann. vol %,Sharpe,Sharpe SE (Lo),Sharpe SE (HAC),NW t-stat,Sortino,Max DD %,Calmar,Total return %,Years,Market beta,Market corr,Trades,Hit rate %,Avg days held
S0 Distance (Gatev 2006),-0.090,1.820,-0.050,0.250,0.230,-0.210,-0.070,-5.780,-0.020,-1.620,15.710,0.020,0.160,651.000,53.460,54.990
S1 Cointegration z-score,0.330,1.690,0.190,0.250,0.220,0.860,0.280,-3.940,0.080,5.000,15.710,0.020,0.230,424.000,62.740,47.080
S2 OU s-score (Avellaneda-Lee 2010),0.100,1.800,0.060,0.250,0.220,0.250,0.080,-4.160,0.020,1.320,15.710,0.030,0.240,750.000,68.930,31.590
S3 OU optimal bands (Bertram 2010),0.120,1.930,0.060,0.250,0.220,0.270,0.090,-4.840,0.020,1.560,15.710,0.020,0.220,618.000,63.920,50.810


Ann. return %  Sharpe  \
S0 Distance (Gatev 2006)            2011-2021               0.000   0.000   
                                    2022-2026 test         -0.300  -0.160   
S1 Cointegration z-score            2011-2021               0.040   0.020   
                                    2022-2026 test          0.990   0.690   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2021              -0.020  -0.010   
                                    2022-2026 test          0.370   0.240   
S3 OU optimal bands (Bertram 2010)  2011-2021               0.090   0.040   
                                    2022-2026 test          0.180   0.100   

                                                    Sharpe SE (Lo)  Max DD %  
S0 Distance (Gatev 2006)            2011-2021                0.300    -5.780  
                                    2022-2026 test           0.460    -3.340  
S1 Cointegration z-score            2011-2021                0.300    -3.940  
                                    2022-2026 test           0.460    -2.330  
S2 OU s-score (Avellaneda-Lee 2010) 2011-2021                0.300    -4.160  
                                    2022-2026 test           0.460    -2.940  
S3 OU optimal bands (Bertram 2010)  2011-2021                0.300    -4.310  
                                    2022-2026 test           0.460    -3.790

In [26]:
if RUN_HEAVY:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for s, r in res_us.items():
        ax.plot((1 + r.committed).cumprod(), lw=1.2, color=COLORS[s], label=C.STRATEGY_LABELS[s])
    ax.axvspan(pd.Timestamp(C.TEST_START), px_us.index[-1], color='grey', alpha=0.12, label='Test block')
    ax.set_ylabel('Wealth on committed capital (start = 1)'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e5_cumulative')

In [27]:
if RUN_HEAVY:
    # M4: why the committed-capital returns are small. Few pairs pass, so most of the K = 20 slots are idle,
    # and costs take most of the gross edge.
    diag_us = pd.DataFrame([s['diag'] for s in sel_us])
    print(f"Per window: {diag_us.n_tested.mean():.0f} pairs tested, {diag_us.n_eg.mean():.1f} pass Engle-Granger "
          f"(chance alone at 5%: up to {0.05 * diag_us.n_tested.mean():.1f}), {diag_us.n_bh.mean():.1f} survive Benjamini-Hochberg, "
          f"{diag_us.n_filtered.mean():.1f} pass all filters (of K = {C.TOP_K} slots)")
    e5_diag = pd.DataFrame({
        C.STRATEGY_LABELS[s]: {
            'Sharpe, 10 bp (committed)': compute_metrics(r.committed)['Sharpe'],
            'Sharpe, 0 bp (committed)': compute_metrics(simulate(s, px_us, sel_us, 0).committed)['Sharpe'],
            'Sharpe, 10 bp (employed)': compute_metrics(r.employed)['Sharpe'],
            'Trades per pair-window': len(r.trades) / max(len(r.pairs), 1),
        } for s, r in res_us.items()}).T
    e5_diag.to_csv('results/e5_diagnostics.csv'); display(e5_diag.round(2))

Per window: 196 pairs tested, 8.4 pass Engle-Granger (chance alone at 5%: up to 9.8), 0.6 survive Benjamini-Hochberg, 8.1 pass all filters (of K = 20 slots)


,"Sharpe, 10 bp (committed)","Sharpe, 0 bp (committed)","Sharpe, 10 bp (employed)",Trades per pair-window
S0 Distance (Gatev 2006),-0.050,0.180,-0.080,1.020
S1 Cointegration z-score,0.190,0.350,0.010,1.640
S2 OU s-score (Avellaneda-Lee 2010),0.060,0.320,-0.220,2.900
S3 OU optimal bands (Bertram 2010),0.060,0.230,-0.140,2.390


**Is any difference real? (M1, T40)** A Sharpe ratio estimated from 16 years of data has a standard error of about $1/\sqrt{16} = 0.25$, so differences between rules can easily be noise. We test each OU rule against the distance baseline with a circular block bootstrap of the Sharpe difference (21-day blocks, the same days for both rules; Ledoit & Wolf 2008), and correct the three p-values with Holm's method because we run three tests.

In [28]:
if RUN_HEAVY:
    from src.metrics import sharpe_diff_bootstrap
    rows = {}
    for s in ['zscore', 'sscore', 'bertram']:
        for blk, sl in [('2011-2026', slice(None)), ('2022-2026 test', slice(C.TEST_START, None))]:
            rows[(C.STRATEGY_LABELS[s], blk)] = sharpe_diff_bootstrap(res_us[s].committed[sl], res_us['distance'].committed[sl])
    sig = pd.DataFrame(rows).T
    for blk in sig.index.get_level_values(1).unique():          # Holm step-down over the three rules
        p = sig.xs(blk, level=1)['p_value'].sort_values()
        adj = np.minimum(1, np.maximum.accumulate(p.values * np.arange(len(p), 0, -1)))
        for name, a in zip(p.index, adj):
            sig.loc[(name, blk), 'p_Holm'] = a
    sig.to_csv('results/significance_vs_distance.csv'); display(sig.round(3))

sharpe_1  sharpe_2  diff  \
S1 Cointegration z-score            2011-2026          0.192    -0.048 0.240   
                                    2022-2026 test     0.687    -0.163 0.850   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026          0.055    -0.048 0.104   
                                    2022-2026 test     0.240    -0.163 0.403   
S3 OU optimal bands (Bertram 2010)  2011-2026          0.061    -0.048 0.109   
                                    2022-2026 test     0.105    -0.163 0.268   

                                                    ci_low  ci_high  p_value  \
S1 Cointegration z-score            2011-2026       -0.255    0.762    0.344   
                                    2022-2026 test  -0.213    1.934    0.115   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026       -0.389    0.617    0.679   
                                    2022-2026 test  -0.650    1.484    0.458   
S3 OU optimal bands (Bertram 2010)  2011-2026       -0.390    0.611    0.661   
                                    2022-2026 test  -0.859    1.413    0.643   

                                                    p_Holm  
S1 Cointegration z-score            2011-2026        1.000  
                                    2022-2026 test   0.344  
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026        1.000  
                                    2022-2026 test   0.917  
S3 OU optimal bands (Bertram 2010)  2011-2026        1.000  
                                    2022-2026 test   0.917

**How big is look-ahead bias here? (M1)** The same selected pairs, traded three ways: honestly; with execution at the signal's own close (lag 0); and with $\beta$, $\mu$ and the OU parameters re-estimated on the trading window itself, i.e. on prices not yet seen (`peeking_selections`, demonstration only).

In [29]:
if RUN_HEAVY:
    from src.backtest import peeking_selections
    peek = peeking_selections(sel_us, px_us)
    rows = {}
    for s in STRATS:
        rows[(C.STRATEGY_LABELS[s], 'honest')] = compute_metrics(res_us[s].committed)
        rows[(C.STRATEGY_LABELS[s], 'lag 0')] = compute_metrics(simulate(s, px_us, sel_us, C.COST_BPS['us'], lag=0).committed)
        if s != 'distance':
            rows[(C.STRATEGY_LABELS[s], 'peeking')] = compute_metrics(simulate(s, px_us, peek, C.COST_BPS['us']).committed)
    bias = pd.DataFrame(rows).T[['Ann. return %', 'Sharpe', 'NW t-stat', 'Max DD %']]
    bias.to_csv('results/m1_lookahead_bias.csv'); display(bias.round(2))

Ann. return %  Sharpe  NW t-stat  \
S0 Distance (Gatev 2006)            honest          -0.090  -0.050     -0.210   
                                    lag 0            0.120   0.060      0.270   
S1 Cointegration z-score            honest           0.330   0.190      0.860   
                                    lag 0            0.430   0.240      1.100   
                                    peeking          2.690   2.840      9.420   
S2 OU s-score (Avellaneda-Lee 2010) honest           0.100   0.060      0.250   
                                    lag 0            0.160   0.090      0.390   
                                    peeking          5.060   3.600     13.330   
S3 OU optimal bands (Bertram 2010)  honest           0.120   0.060      0.270   
                                    lag 0            0.160   0.080      0.360   
                                    peeking          7.410   3.650     14.780   

                                             Max DD %  
S0 Distance (Gatev 2006)            honest     -5.780  
                                    lag 0      -5.140  
S1 Cointegration z-score            honest     -3.940  
                                    lag 0      -3.970  
                                    peeking    -1.390  
S2 OU s-score (Avellaneda-Lee 2010) honest     -4.160  
                                    lag 0      -4.410  
                                    peeking    -1.200  
S3 OU optimal bands (Bertram 2010)  honest     -4.840  
                                    lag 0      -4.270  
                                    peeking    -1.820

**E5 interpretation (M4).** US, Jan 2011 - Sep 2026, 10 bp per leg, committed capital (results/table1_us.csv, results/e5_diagnostics.csv).

- **Which rule wins.** S1 z-score has the best numbers (0.33% a year, Sharpe 0.19, max drawdown -3.9%), ahead of S3 Bertram (0.12%, 0.06), S2 s-score (0.10%, 0.06) and the S0 distance baseline (-0.09%, -0.05). The ranking holds in the 2022-2026 test block (S1 Sharpe 0.69, S0 -0.16), but none of the gaps is significant: M1's block bootstrap gives Holm-adjusted p-values of 0.34 or more against S0.
- **Is any Sharpe ratio significant?** No. Every Sharpe ratio is within one Lo standard error (0.25 over 15.7 years) of zero, and every Newey-West t-statistic is below 1 (S1: 0.86). We cannot reject a true Sharpe ratio of zero for any rule.
- **Market neutrality.** Yes: betas to SPY are 0.02-0.03 and correlations 0.16-0.24, so the returns are not disguised market exposure.
- **Why the returns are so small.** Only about 8 of the 196 candidate pairs pass Engle-Granger per window, no more than the 10 a 5% test would flag by chance alone, and Benjamini-Hochberg keeps fewer than one. So 12 of the 20 capital slots sit idle, and many traded pairs are probably false positives (consistent with the low ADF power in E2a). Before costs S1-S3 earn Sharpe 0.23-0.35; 10 bp per leg removes most of that. S2 trades most (750 round trips, 32 days held on average) and so pays the most cost, despite the highest hit rate (69%).

In [30]:
if RUN_HEAVY:
    # E5b (M4): trade-level evidence. One Sharpe ratio over 16 years has little power; ~1,800 round trips have more.
    # t-statistics are clustered by walk-forward window, because trades in the same window share market moves.
    import statsmodels.api as sm

    def clustered_mean(y, groups):
        fit = sm.OLS(np.asarray(y, float), np.ones(len(y))).fit(cov_type='cluster', cov_kwds={'groups': np.asarray(groups)})
        return float(fit.params[0]), float(fit.tvalues[0])

    last_day = {sel['window'].i: px_us.index[sel['window'].t1 - 1] for sel in sel_us}
    rows, exits = {}, {}
    for s in STRATS:
        for cost in [0, C.COST_BPS['us']]:
            t = (res_us[s] if cost else simulate(s, px_us, sel_us, 0)).trades.copy()
            t['window'] = t['pair'].str.slice(1, 3).astype(int)
            m, tm = clustered_mean(1e4 * t['pnl'], t['window'])
            h, th = clustered_mean((t['pnl'] > 0) - 0.5, t['window'])
            rows[(C.STRATEGY_LABELS[s], f'{cost} bp')] = {
                'Trades': len(t), 'Mean P&L per trade (bp)': m, 't (mean)': tm,
                'Hit rate %': 100 * (h + 0.5), 't (hit vs 50%)': th}
            if cost and s != 'distance':
                # Where the money goes: exits by the rule vs positions force-closed at the window's last day
                t['exit'] = np.where(t['close'] == t['window'].map(last_day), 'forced at window end', 'rule exit')
                for e, g in t.groupby('exit'):
                    exits[(C.STRATEGY_LABELS[s], e)] = {
                        'Share of trades %': 100 * len(g) / len(t), 'Hit rate %': 100 * (g['pnl'] > 0).mean(),
                        'Mean P&L per trade (bp)': 1e4 * g['pnl'].mean(), 'Total P&L (bp of pair capital)': 1e4 * g['pnl'].sum()}
    e5_trades = pd.DataFrame(rows).T
    e5_exits = pd.DataFrame(exits).T
    e5_trades.to_csv('results/e5_trades.csv'); e5_exits.to_csv('results/e5_exits.csv')
    hl = pd.cut(res_us['sscore'].pairs['half_life'], [0, 10, 21, 63]).value_counts().sort_index()
    print('Selected pair-windows by formation half-life (days):', dict(zip(hl.index.astype(str), hl.values)))
    display(e5_trades.round(2)); display(e5_exits.round(1))

Selected pair-windows by formation half-life (days): {'(0, 10]': np.int64(242), '(10, 21]': np.int64(17), '(21, 63]': np.int64(0)}


Trades  Mean P&L per trade (bp)  \
S0 Distance (Gatev 2006)            0 bp    651.000                   15.770   
                                    10 bp   651.000                   -4.230   
S1 Cointegration z-score            0 bp    424.000                   44.100   
                                    10 bp   424.000                   24.100   
S2 OU s-score (Avellaneda-Lee 2010) 0 bp    750.000                   24.170   
                                    10 bp   750.000                    4.170   
S3 OU optimal bands (Bertram 2010)  0 bp  1,577.000                    9.000   
                                    10 bp   618.000                    5.940   

                                           t (mean)  Hit rate %  \
S0 Distance (Gatev 2006)            0 bp      0.930      56.530   
                                    10 bp    -0.250      53.460   
S1 Cointegration z-score            0 bp      2.010      64.860   
                                    10 bp     1.100      62.740   
S2 OU s-score (Avellaneda-Lee 2010) 0 bp      1.860      71.470   
                                    10 bp     0.320      68.930   
S3 OU optimal bands (Bertram 2010)  0 bp      1.420      68.170   
                                    10 bp     0.370      63.920   

                                           t (hit vs 50%)  
S0 Distance (Gatev 2006)            0 bp            2.790  
                                    10 bp           1.470  
S1 Cointegration z-score            0 bp            7.460  
                                    10 bp           5.710  
S2 OU s-score (Avellaneda-Lee 2010) 0 bp           12.650  
                                    10 bp          10.240  
S3 OU optimal bands (Bertram 2010)  0 bp           14.640  
                                    10 bp           9.130

Share of trades %  \
S1 Cointegration z-score            forced at window end             45.300   
                                    rule exit                        54.700   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end             28.400   
                                    rule exit                        71.600   
S3 OU optimal bands (Bertram 2010)  forced at window end             41.900   
                                    rule exit                        58.100   

                                                          Hit rate %  \
S1 Cointegration z-score            forced at window end      20.800   
                                    rule exit                 97.400   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end      15.500   
                                    rule exit                 90.100   
S3 OU optimal bands (Bertram 2010)  forced at window end      17.400   
                                    rule exit                 97.500   

                                                          Mean P&L per trade (bp)  \
S1 Cointegration z-score            forced at window end                 -278.400   
                                    rule exit                             274.400   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end                 -340.400   
                                    rule exit                             140.800   
S3 OU optimal bands (Bertram 2010)  forced at window end                 -329.000   
                                    rule exit                             247.600   

                                                          Total P&L (bp of pair capital)  
S1 Cointegration z-score            forced at window end                     -53,451.000  
                                    rule exit                                 63,669.300  
S2 OU s-score (Avellaneda-Lee 2010) forced at window end                     -72,500.600  
                                    rule exit                                 75,627.500  
S3 OU optimal bands (Bertram 2010)  forced at window end                     -85,218.700  
                                    rule exit                                 88,892.200

**E5b interpretation (M4): the signal is real; broken pairs eat the profit.** (results/e5_trades.csv, results/e5_exits.csv)

- **Mean reversion is detectable at trade level.** S1-S3 win 63-69% of round trips after costs, far above 50% (window-clustered t = 5.7-10.2). Before costs S1 earns 44 bp per trade (t = 2.0) and S2 24 bp (t = 1.9). A round trip costs about 20 bp of pair capital at 10 bp per leg, which removes half of S1's edge and nearly all of S2's.
- **Where the money goes.** Trades closed by the rule win 90-97% of the time (+141 to +274 bp on average). The 28-45% of trades still open when the window ends, i.e. spreads that did not revert within six months, lose 278-340 bp on average and cancel almost all of those gains. Part of the rule-exit win rate is mechanical (a rule exit is a move back toward the mean), so the informative numbers are how often trades get stuck and how much they lose.
- **Link to E2a and Engle-Granger.** Every selected pair has a formation half-life of 21 days or less (242 of 259 at 10 days or less), as E2a's power curve predicts. The losses therefore come not from slow pairs but from pairs whose relationship broke, consistent with Engle-Granger passing about as many pairs as chance would (slide 5; M2's E9 measures this directly).
- Not done, on purpose: no time stop or stop-loss was added after seeing these numbers, since that would be tuning on results (M3's E7 tests the 4-sigma stop-loss as a sensitivity check).

**E5c (M4): time stop at 3 half-lives. Hypothesis written on 12 Nov 2026, before the variant was run.**

E5b showed that trades still open at the end of the window (spreads that never reverted) cause the losses. Under the OU model the expected deviation decays as $e^{-\kappa t}$, so after 3 half-lives only $2^{-3} = 12.5\%$ of it should remain. A spread still beyond the exit band after that long is evidence that the pair no longer behaves like an OU process.

- **Rule.** For S1 and S2, close a position held for more than $3 \times$ the pair's formation half-life (rounded up, in trading days) and block re-entry until the score is back inside the entry band (as after a stop-loss).
- **Fixed in advance.** The multiple 3 comes from the OU decay above, not from our data; no other value is run. S0 has no half-life and S3 belongs to M5, so both are left unchanged.
- **Hypothesis.** The time stop reduces the losses on trades that never revert and raises the net Sharpe ratio of S1 and S2. It may also cut slow winners, so the sign of the net effect is not known in advance.
- **Reporting.** Run once on the US sample, reported whatever the outcome, next to the unchanged headline (Table 1 keeps the paper rules). India, not yet run, is the confirmation sample.

In [31]:
if RUN_HEAVY:
    # E5c (M4): the pre-registered time stop, run once. Headline rules (res_us) are unchanged.
    from src.metrics import sharpe_diff_bootstrap
    ts = {s: simulate(s, px_us, sel_us, C.COST_BPS['us'], params={'time_stop': C.TIME_STOP_HALF_LIVES})
          for s in ['zscore', 'sscore']}
    last_day = {sel['window'].i: px_us.index[sel['window'].t1 - 1] for sel in sel_us}
    rows, exits = {}, {}
    for s, r_ts in ts.items():
        for name, r in [('headline', res_us[s]), (f'time stop {C.TIME_STOP_HALF_LIVES} half-lives', r_ts)]:
            for blk, sl in [('2011-2026', slice(None)), ('2022-2026 test', slice(C.TEST_START, None))]:
                m = compute_metrics(r.committed[sl])
                rows[(C.STRATEGY_LABELS[s], name, blk)] = {k: m[k] for k in ['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'NW t-stat', 'Max DD %']}
            t = r.trades.copy()
            t['window'] = t['pair'].str.slice(1, 3).astype(int)
            hl = r.pairs.set_index('pair')['half_life']
            limit = np.ceil(C.TIME_STOP_HALF_LIVES * t['pair'].map(hl))
            forced = t['close'] == t['window'].map(last_day)
            t['exit'] = np.where(forced, 'forced at window end',
                                 np.where((t['days'] >= limit) & (name != 'headline'), 'time stop', 'rule exit'))
            for e, g in t.groupby('exit'):
                exits[(C.STRATEGY_LABELS[s], name, e)] = {
                    'Trades': len(g), 'Share %': 100 * len(g) / len(t), 'Hit rate %': 100 * (g['pnl'] > 0).mean(),
                    'Mean P&L per trade (bp)': 1e4 * g['pnl'].mean(), 'Total P&L (bp)': 1e4 * g['pnl'].sum()}
        for blk, sl in [('2011-2026', slice(None)), ('2022-2026 test', slice(C.TEST_START, None))]:
            b = sharpe_diff_bootstrap(r_ts.committed[sl], res_us[s].committed[sl])
            rows[(C.STRATEGY_LABELS[s], 'time stop minus headline', blk)] = {
                'Sharpe': b['diff'], 'Sharpe SE (Lo)': np.nan, 'NW t-stat': np.nan,
                'Ann. return %': np.nan, 'Max DD %': np.nan, 'bootstrap p': b['p_value']}
    e5c = pd.DataFrame(rows).T
    e5c_exits = pd.DataFrame(exits).T
    e5c.to_csv('results/e5c_time_stop.csv'); e5c_exits.to_csv('results/e5c_exits.csv')
    display(e5c.round(3)); display(e5c_exits.round(1))

Ann. return %  \
S1 Cointegration z-score            headline                 2011-2026               0.325   
                                                             2022-2026 test          0.985   
                                    time stop 3 half-lives   2011-2026              -0.051   
                                                             2022-2026 test          0.760   
                                    time stop minus headline 2011-2026                 NaN   
                                                             2022-2026 test            NaN   
S2 OU s-score (Avellaneda-Lee 2010) headline                 2011-2026               0.100   
                                                             2022-2026 test          0.366   
                                    time stop 3 half-lives   2011-2026              -0.285   
                                                             2022-2026 test          0.163   
                                    time stop minus headline 2011-2026                 NaN   
                                                             2022-2026 test            NaN   

                                                                             Sharpe  \
S1 Cointegration z-score            headline                 2011-2026        0.192   
                                                             2022-2026 test   0.687   
                                    time stop 3 half-lives   2011-2026       -0.042   
                                                             2022-2026 test   0.797   
                                    time stop minus headline 2011-2026       -0.234   
                                                             2022-2026 test   0.110   
S2 OU s-score (Avellaneda-Lee 2010) headline                 2011-2026        0.055   
                                                             2022-2026 test   0.240   
                                    time stop 3 half-lives   2011-2026       -0.221   
                                                             2022-2026 test   0.169   
                                    time stop minus headline 2011-2026       -0.277   
                                                             2022-2026 test  -0.072   

                                                                             Sharpe SE (Lo)  \
S1 Cointegration z-score            headline                 2011-2026                0.252   
                                                             2022-2026 test           0.461   
                                    time stop 3 half-lives   2011-2026                0.252   
                                                             2022-2026 test           0.461   
                                    time stop minus headline 2011-2026                  NaN   
                                                             2022-2026 test             NaN   
S2 OU s-score (Avellaneda-Lee 2010) headline                 2011-2026                0.252   
                                                             2022-2026 test           0.460   
                                    time stop 3 half-lives   2011-2026                0.252   
                                                             2022-2026 test           0.460   
                                    time stop minus headline 2011-2026                  NaN   
                                                             2022-2026 test             NaN   

                                                                             NW t-stat  \
S1 Cointegration z-score            headline                 2011-2026           0.856   
                                                             2022-2026 test      1.612   
                                    time stop 3 half-lives   2011-2026          -0.181   
                                                             2022-2026 test      1.993   
                                    

Trades  \
S1 Cointegration z-score            headline               forced at window end 192.000   
                                                           rule exit            232.000   
                                    time stop 3 half-lives forced at window end  55.000   
                                                           rule exit            146.000   
                                                           time stop            391.000   
S2 OU s-score (Avellaneda-Lee 2010) headline               forced at window end 213.000   
                                                           rule exit            537.000   
                                    time stop 3 half-lives forced at window end  47.000   
                                                           rule exit            415.000   
                                                           time stop            391.000   

                                                                                 Share %  \
S1 Cointegration z-score            headline               forced at window end   45.300   
                                                           rule exit              54.700   
                                    time stop 3 half-lives forced at window end    9.300   
                                                           rule exit              24.700   
                                                           time stop              66.000   
S2 OU s-score (Avellaneda-Lee 2010) headline               forced at window end   28.400   
                                                           rule exit              71.600   
                                    time stop 3 half-lives forced at window end    5.500   
                                                           rule exit              48.700   
                                                           time stop              45.800   

                                                                                 Hit rate %  \
S1 Cointegration z-score            headline               forced at window end      20.800   
                                                           rule exit                 97.400   
                                    time stop 3 half-lives forced at window end      43.600   
                                                           rule exit                 96.600   
                                                           time stop                 36.300   
S2 OU s-score (Avellaneda-Lee 2010) headline               forced at window end      15.500   
                                                           rule exit                 90.100   
                                    time stop 3 half-lives forced at window end      38.300   
                                                           rule exit                 87.700   
                                                           time stop                 24.000   

                                                                                 Mean P&L per trade (bp)  \
S1 Cointegration z-score            headline               forced at window end                 -278.400   
                                                           rule exit                             274.400   
                                    time stop 3 half-lives forced at window end                  -52.600   
                                                           rule exit                             256.700   
                                                           time stop                             -92.600   
S2 OU s-score (Avellaneda-Lee 2010) headline               forced at window end                 -340.400   
                                                           rule exit                             140.800   
                                    time stop 3 half-lives forced at window end                  -78.800   
                                                

**E5c result (M4): the hypothesis is rejected; the time stop makes both rules worse.** (results/e5c_time_stop.csv, results/e5c_exits.csv)

- **Performance, 2011-2026.** S1's Sharpe ratio falls from 0.19 to -0.04 and S2's from 0.06 to -0.22; maximum drawdowns roughly double (S1 -3.9% to -7.6%, S2 -4.2% to -8.6%). The block-bootstrap differences (-0.23 and -0.28) are not significant (p = 0.13 and 0.11), so the honest reading is "no improvement, probably harm". In the 2022-2026 test block S1 rises slightly (0.69 to 0.80, p = 0.70) and S2 falls (0.24 to 0.17).
- **What it did.** It removed the losses it was aimed at: the force-closed losses shrink from about 53,000 to 2,900 bp (S1) and from 72,500 to 3,700 bp (S2). But it fired on 66% of S1 trades and 46% of S2 trades, and those trades lost 93 and 141 bp on average. It cut many slow trades that would have reverted later (rule-exit gains fall by about a third) and added round trips (S1 592 trades instead of 424), so costs rose too.
- **Why the theory-based clock was too short.** The clock uses the fitted half-life, and E1 shows $\hat\kappa$ is biased upward on a 252-day window (by about 20% at $\kappa = 25$ and more for slower pairs), so fitted half-lives are too short and the stop fires early. A stopped trade is also usually still far from the mean, so it locks in a loss.
- **India (confirmation sample, run 13 Nov).** The same rule on India also lowers the Sharpe ratio: S1 0.38 to -0.08 and S2 0.32 to -0.03 (results/e8_india_detail.csv), so the rejection replicates.
- **Conclusion.** The headline keeps the paper rules. Broken pairs are better addressed by selection (fewer false positives, E9) than by exiting early; we do not try other multiples, since that would be tuning on results.

## 9. E6 transaction costs  (Owner: M3)  HEAVY
Costs enter linearly, so the break-even cost per leg is where the annual return reaches zero.

In [32]:
if RUN_HEAVY:
    # E6 (M3): cost sweep. S0-S2 trade the same way at every cost, so their return is linear in cost;
    # S3's Bertram bands narrow as cost falls (1,577 trades at 0 bp vs 618 at 10 bp), so it is not.
    # Break-even is therefore read off where each curve crosses zero (linear interpolation), not a line fit.
    cost_grid = [0, 5, 10, 15, 20, 30]
    e6 = pd.DataFrame([dict(strategy=s, cost_bps=c, **compute_metrics(simulate(s, px_us, sel_us, c).committed))
                       for s in STRATS for c in cost_grid])
    e6.to_csv('results/e6_costs.csv', index=False)

    def crossing(costs, values):
        """First cost at which values change sign from >= 0 to < 0 (linear interpolation)."""
        for c0, c1, v0, v1 in zip(costs[:-1], costs[1:], values[:-1], values[1:]):
            if v0 >= 0 > v1:
                return c0 + v0 * (c1 - c0) / (v0 - v1)
        return 0.0 if values[0] < 0 else np.nan   # loses even at 0 bp / never breaks even in the grid

    breakeven = {}
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for s in STRATS:
        d = e6[e6.strategy == s].sort_values('cost_bps')
        ax.plot(d.cost_bps, d['Sharpe'], marker='o', color=COLORS[s], label=C.STRATEGY_LABELS[s])
        breakeven[C.STRATEGY_LABELS[s]] = crossing(d.cost_bps.values, d['Ann. return %'].values)
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(C.COST_BPS['us'], color='grey', ls=':', lw=0.8)   # headline cost, 10 bp
    ax.set_xlabel('Cost per leg per transaction (bp)'); ax.set_ylabel('Sharpe ratio'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e6_costs')

    breakeven = pd.Series(breakeven, name='Break-even cost (bp per leg)')
    breakeven.to_csv('results/e6_breakeven.csv')
    display(e6.pivot(index='cost_bps', columns='strategy', values='Sharpe').round(2))
    display(breakeven.round(1))

strategy,bertram,distance,sscore,zscore
cost_bps,,,,
0,0.230,0.180,0.320,0.350
5,0.200,0.070,0.190,0.270
10,0.060,-0.050,0.060,0.190
15,-0.050,-0.160,-0.080,0.110
20,-0.080,-0.270,-0.210,0.030
30,-0.140,-0.500,-0.470,-0.130


S0 Distance (Gatev 2006)               7.900
S1 Cointegration z-score              22.000
S2 OU s-score (Avellaneda-Lee 2010)   12.100
S3 OU optimal bands (Bertram 2010)    12.900
Name: Break-even cost (bp per leg), dtype: float64

**E6 (M3): costs decide the result.** Before costs all four rules have positive Sharpe ratios (0.18–0.35). Break-even cost per leg: S1 22 bp, S3 13 bp, S2 12 bp, S0 8 bp (results/e6_breakeven.csv), so at 10 bp only S1 keeps a clear margin. S0–S2 trade the same way at every cost, so their Sharpe falls linearly, steepest for S2 because it trades most. S3 is the exception: Bertram's band widens with cost, so it trades less (1,577 trades at 0 bp, 618 at 10 bp) and degrades slowest (−0.14 at 30 bp vs −0.47 for S2).

## 10. E7 parameter sensitivity  (Owner: M3)  HEAVY
Reported as analysis only; the headline result keeps the paper thresholds.

In [35]:
if RUN_HEAVY:
    # E7 (M3): sensitivity of S2 (s-score) to its thresholds, formation length, K and a stop-loss.
    # Analysis only: the headline keeps the paper thresholds (Avellaneda & Lee 2010).
    entries, exits = [1.0, 1.25, 1.5, 2.0, 2.5], [0.0, 0.25, 0.5, 0.75]
    grid = pd.DataFrame(index=entries, columns=exits, dtype=float)
    for e in entries:
        for x in exits:
            p = dict(open_long=-e, open_short=e, close_long=-x, close_short=x)
            grid.loc[e, x] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], params=p).committed)['Sharpe']
    grid.index.name, grid.columns.name = 'Entry |s|', 'Exit |s|'
    grid.to_csv('results/e7_threshold_grid.csv')

    fig, ax = plt.subplots(figsize=(5, 3.6))
    im = ax.imshow(grid.values.astype(float), cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(exits)), exits); ax.set_yticks(range(len(entries)), entries)
    for i in range(len(entries)):
        for j in range(len(exits)):
            ax.text(j, i, f'{grid.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_xlabel('Exit |s|'); ax.set_ylabel('Entry |s|'); ax.grid(False)
    fig.colorbar(im, label='Sharpe ratio (S2, 10 bp)')
    savefig(fig, 'fig_e7_heatmap')
    display(grid.round(2))

    # Robustness rows. Different formation lengths start trading on different dates,
    # so every row is evaluated on the period common to all runs.
    runs = {'Headline (paper thresholds, 252 d, K = 20)': res_us['sscore']}
    for f_days in [126, 504]:
        sel_f = run_selection('us', px_us, sectors('us'), formation=f_days)
        runs[f'Formation {f_days} d'] = simulate('sscore', px_us, sel_f, C.COST_BPS['us'])
    for k in [5, 10]:
        runs[f'K = {k}'] = simulate('sscore', px_us, sel_us, C.COST_BPS['us'], k=k)
    runs['Stop-loss |s| > 4'] = simulate('sscore', px_us, sel_us, C.COST_BPS['us'], stop=4.0)

    start = max(r.committed.index[0] for r in runs.values())
    end = min(r.committed.index[-1] for r in runs.values())
    print(f'Common evaluation period: {start.date()} to {end.date()}')
    in_period = lambda t: t[(t['open'] >= start) & (t['open'] <= end)]
    e7 = pd.DataFrame({name: compute_metrics(r.committed[start:end], trades=in_period(r.trades))
                       for name, r in runs.items()}).T
    e7 = e7[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %', 'Trades', 'Hit rate %']]
    e7.to_csv('results/e7_robustness.csv'); display(e7.round(2))

Exit |s|,0.000,0.250,0.500,0.750
Entry |s|,,,,
1.000,0.110,0.050,0.060,-0.010
1.250,0.100,0.070,0.080,0.030
1.500,0.140,0.090,0.120,0.070
2.000,0.170,0.130,0.150,0.150
2.500,0.170,0.140,0.160,0.190


Common evaluation period: 2012-01-03 to 2026-09-29


,Ann. return %,Sharpe,Sharpe SE (Lo),Max DD %,Trades,Hit rate %
"Headline (paper thresholds, 252 d, K = 20)",0.090,0.050,0.260,-4.160,677.000,69.420
Formation 126 d,0.050,0.030,0.260,-6.220,867.000,66.900
Formation 504 d,0.320,0.190,0.260,-4.620,606.000,69.140
K = 5,-0.190,-0.050,0.260,-12.630,385.000,71.170
K = 10,0.090,0.030,0.260,-6.350,626.000,70.290
Stop-loss |s| > 4,-0.090,-0.090,0.260,-4.530,704.000,63.210


**E7 (M3): no setting rescues S2 significantly** (common period 2012–2026, Lo SE ≈ 0.26).
- **Thresholds:** Sharpe rises with the entry band (≈0.05 at |s| = 1 to ≈0.17 at 2–2.5) because wider bands trade less (the E6 cost effect); the exit level matters little. The grid spans less than one SE, so picking the best cell would be tuning on results.
- **Formation length:** 504 days gives 0.19 vs 0.05 (252 d) and 0.03 (126 d), consistent with E1 and E2a, but one comparison within noise: a hypothesis, not a result.
- **K and stop-loss:** top 5 or 10 pairs give −0.05 and 0.03 (drawdown −12.6% at K = 5), so the ranking adds nothing; a stop-loss at |s| > 4 gives −0.09 and cuts the hit rate from 69% to 63%, like the E5c time stop.

## 11. E8 markets, sectors and sub-periods  (Owners: M2 India selection, M5 India table, M3 US sector and sub-period tables)  HEAVY

In [36]:
if RUN_HEAVY:
    sel_in = run_selection('india', px_in, sectors('india'))
    res_in = {s: simulate(s, px_in, sel_in, C.COST_BPS['india'], label=s) for s in STRATS}
    table_in = metrics_table(res_in.values(), bench_in).rename(index=C.STRATEGY_LABELS)
    table_in.to_csv('results/table_india.csv'); display(table_in.round(2))

    # E8 US (M3): Sharpe by sector. Each sector's sub-portfolio is the sum of its selected
    # pairs' returns (Sharpe is scale-free, so the capital base does not matter).
    by_sector = {}
    for s, r in res_us.items():
        for sec, cols in r.pairs.groupby('sector')['pair']:
            by_sector.setdefault(sec, {})[C.STRATEGY_LABELS[s]] = \
                compute_metrics(r.pair_returns[list(cols)].sum(axis=1))['Sharpe']
    by_sector = pd.DataFrame(by_sector).T
    by_sector.insert(0, 'Pair-windows (S1-S3)', res_us['zscore'].pairs.groupby('sector').size())
    by_sector.insert(1, 'Pair-windows (S0)', res_us['distance'].pairs.groupby('sector').size())
    by_sector = by_sector.fillna({'Pair-windows (S1-S3)': 0, 'Pair-windows (S0)': 0})
    by_sector.index.name = 'Sector'
    by_sector.to_csv('results/e8_sectors.csv'); display(by_sector.round(2))

    # E8 US (M3): Sharpe by sub-period, all strategies (Table 2)
    sub = pd.DataFrame({name: {C.STRATEGY_LABELS[s]: compute_metrics(res.committed[a:b])['Sharpe']
                               for s, res in res_us.items()}
                        for name, (a, b) in C.SUB_PERIODS.items()})
    sub.to_csv('results/e8_subperiods.csv'); display(sub.round(2))

  india window  0 2013-01-10:   0/55 pass EG,  0 pass filters
  india window  1 2013-07-12:   2/55 pass EG,  2 pass filters
  india window  2 2014-01-16:   2/55 pass EG,  2 pass filters
  india window  3 2014-07-21:  10/55 pass EG, 10 pass filters
  india window  4 2015-01-29:   8/55 pass EG,  8 pass filters
  india window  5 2015-08-03:   4/55 pass EG,  4 pass filters
  india window  6 2016-02-08:   3/55 pass EG,  3 pass filters
  india window  7 2016-08-11:   3/55 pass EG,  3 pass filters
  india window  8 2017-02-15:   2/55 pass EG,  2 pass filters
  india window  9 2017-08-21:   2/55 pass EG,  2 pass filters
  india window 10 2018-02-21:   0/55 pass EG,  0 pass filters
  india window 11 2018-08-24:   5/55 pass EG,  3 pass filters
  india window 12 2019-02-28:   2/55 pass EG,  2 pass filters
  india window 13 2019-09-09:   3/55 pass EG,  3 pass filters
  india window 14 2020-03-13:   1/55 pass EG,  1 pass filters
  india window 15 2020-09-15:   6/55 pass EG,  6 pass filters
  india 

,Ann. return %,Ann. vol %,Sharpe,Sharpe SE (Lo),Sharpe SE (HAC),NW t-stat,Sortino,Max DD %,Calmar,Total return %,Years,Market beta,Market corr,Trades,Hit rate %,Avg days held
S0 Distance (Gatev 2006),0.430,2.700,0.160,0.270,0.260,0.610,0.230,-4.600,0.090,5.410,13.430,0.010,0.050,472.000,53.810,56.880
S1 Cointegration z-score,0.410,1.070,0.380,0.270,0.250,1.510,0.600,-3.250,0.120,5.530,13.430,0.000,0.000,121.000,67.770,47.010
S2 OU s-score (Avellaneda-Lee 2010),0.370,1.150,0.320,0.270,0.240,1.330,0.500,-2.510,0.150,5.000,13.430,-0.000,-0.000,234.000,68.380,29.620
S3 OU optimal bands (Bertram 2010),0.430,1.340,0.320,0.270,0.240,1.310,0.480,-3.600,0.120,5.790,13.430,-0.000,-0.000,203.000,69.460,49.660


,Pair-windows (S1-S3),Pair-windows (S0),S0 Distance (Gatev 2006),S1 Cointegration z-score,S2 OU s-score (Avellaneda-Lee 2010),S3 OU optimal bands (Bertram 2010)
Sector,,,,,,
Banks,55,100,0.100,0.430,0.400,0.500
Consumer staples,29,119,0.200,0.130,0.030,-0.070
Energy,38,41,0.090,0.100,0.010,0.010
Industrials,33,44,-0.040,0.000,-0.060,-0.100
Retail,25,35,-0.100,-0.140,-0.150,-0.210
Technology,21,9,0.050,-0.040,-0.150,-0.080
Utilities,58,292,-0.270,0.150,0.090,0.110


,2010-14,2015-19,2020 (COVID),2021-26
S0 Distance (Gatev 2006),0.200,-0.060,0.080,-0.240
S1 Cointegration z-score,0.020,0.270,-0.030,0.460
S2 OU s-score (Avellaneda-Lee 2010),0.270,-0.150,-0.040,0.090
S3 OU optimal bands (Bertram 2010),0.080,0.240,-0.070,0.020


**E8 US (M3): sectors and sub-periods** (results/e8_sectors.csv, e8_subperiods.csv).
- **No rule is positive in every sub-period.** The distance rule decays from 0.20 (2011–14) to −0.24 (2021–26), as Do & Faff (2010) report; S1 is the most consistent recently (0.27, 0.46); every rule is flat or negative in 2020. ("2010-14" in config starts trading in Jan 2011.)
- **By sector,** banks carry the OU rules (Sharpe 0.40–0.50 for S1–S3, which trade the same bank pairs, so this is one finding, not three); retail and technology lose for every rule; the distance rule loses most in utilities (−0.27), the sector it picks most often (292 of 640 pair-windows).

In [ ]:
if RUN_HEAVY:
    # E8 India (M5): how many pairs pass, the test-block split, and trade-level evidence, as for the US (E5, E5b)
    import statsmodels.api as sm
    diag_in = pd.DataFrame([s['diag'] for s in sel_in])
    print(f"India, per window: {diag_in.n_tested.mean():.0f} pairs tested, {diag_in.n_eg.mean():.1f} pass Engle-Granger "
          f"(chance alone at 5%: up to {0.05 * diag_in.n_tested.mean():.1f}), {diag_in.n_bh.mean():.1f} survive "
          f"Benjamini-Hochberg, {diag_in.n_filtered.mean():.1f} traded; {int((diag_in.n_filtered == 0).sum())} windows with no pair")
    rows = {}
    for s, r in res_in.items():
        t = r.trades.copy()
        t['window'] = t['pair'].str.slice(1, 3).astype(int)
        fit = sm.OLS((t['pnl'] > 0) - 0.5, np.ones(len(t))).fit(cov_type='cluster', cov_kwds={'groups': t['window']})
        row = {'Trades': len(t), 'Hit rate %': 100 * (fit.params.iloc[0] + 0.5), 't (hit vs 50%)': fit.tvalues.iloc[0],
               'Mean P&L per trade (bp)': 1e4 * t['pnl'].mean()}
        for blk, sl in [('2013-2026', slice(None)), ('2013-2021', slice(None, '2021-12-31')),
                        ('2022-2026 test', slice(C.TEST_START, None))]:
            row[f'Sharpe {blk}'] = compute_metrics(r.committed[sl])['Sharpe']
        if s in ('zscore', 'sscore'):
            # E5c confirmation sample, pre-registered by M4 on 12 Nov: the 3-half-life time stop on India
            r_ts = simulate(s, px_in, sel_in, C.COST_BPS['india'], params={'time_stop': C.TIME_STOP_HALF_LIVES})
            row['Sharpe 2013-2026 with E5c time stop'] = compute_metrics(r_ts.committed)['Sharpe']
        rows[C.STRATEGY_LABELS[s]] = row
    e8_india = pd.DataFrame(rows).T
    e8_india.to_csv('results/e8_india_detail.csv'); display(e8_india.round(2))

**E8 India result (M5).** 2013-2026 (first trading window Jan 2013), 15 bp per leg (results/table_india.csv, results/e8_india_detail.csv).

- **Selection is again close to chance.** 3.2 of 55 candidate pairs pass Engle-Granger per window against up to 2.8 by chance; Benjamini-Hochberg keeps 1.0 and 4 windows trade nothing, so only about 3 of the 20 capital slots are used.
- **All four rules are positive, none significantly.** Sharpe ratios are 0.16-0.38 with S1 best, as in the US; the Lo standard error is 0.27 and the largest Newey-West $t$ is 1.51. Market beta is 0.
- **The per-trade signal is stronger than in the US.** After costs S1-S3 win 68-69% of trades ($t$ = 3.6-6.1) and earn 42-90 bp per trade, despite the higher cost, and they stay positive in the 2022-2026 test block (0.28-0.82). This fits a less arbitraged market.
- **E5c confirmation (pre-registered by M4).** The 3-half-life time stop again hurts: S1 0.38 to -0.08 and S2 0.32 to -0.03.
- **Conclusion.** The US pattern replicates: a real per-trade signal, but too few genuine pairs for the portfolio to be significant.

## 12. E9 do ADF-selected pairs stay cointegrated?  (Owner: M2)  HEAVY
For every selected pair we re-test the trading-window spread, built with the formation $\beta$, using ADF (no $\beta$ is re-estimated, so plain ADF critical values are correct here).

In [ ]:
if RUN_HEAVY:
    logp_us = np.log(px_us)
    rows = []
    for sel in sel_us:
        w = sel['window']
        for s in sel['coint']:
            x = (logp_us[s.a] - s.beta * logp_us[s.b] - s.mu).iloc[w.t0:w.t1].dropna().values
            if len(x) > 40:
                rows.append(dict(window=w.i, pair=f'w{w.i:02d}:{s.a}/{s.b}', eg_stat=s.eg_stat,
                                 half_life=s.half_life, still_coint=adf_test(x, maxlag=5)[1] < 0.05))
    e9 = pd.DataFrame(rows).merge(res_us['sscore'].pairs[['pair', 'pnl']], on='pair')
    print(f'Share of selected pairs still cointegrated out of sample: {100 * e9.still_coint.mean():.1f}%')
    e9['decile'] = pd.qcut(e9.eg_stat, 10, labels=False) + 1
    by_dec = e9.groupby('decile').pnl.mean() * 100
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.bar(by_dec.index, by_dec.values, color=COLORS['sscore'])
    ax.set_xlabel('Formation EG statistic decile (1 = most negative)'); ax.set_ylabel('Mean pair P&L per window (%)')
    savefig(fig, 'fig_e9_persistence')
    e9.to_csv('results/e9_persistence.csv', index=False)
    display(e9.groupby('still_coint').pnl.describe())

## 13. E10 Kalman-filter hedge ratio (stretch)  (Owner: M5)  HEAVY
State $(\mu_t,\beta_t)$ follows a random walk, observation $\ln P^A_t = \mu_t + \beta_t\ln P^B_t + e_t$ (Elliott et al. 2005; Chan 2013). Trade the standardised forecast error at $\pm 1$.

In [ ]:
if RUN_HEAVY:
    res_us['kalman'] = simulate('kalman', px_us, sel_us, C.COST_BPS['us'], label='kalman')
    display(metrics_table([res_us['kalman']], bench_us).rename(index=C.STRATEGY_LABELS).round(2))

In [ ]:
if RUN_HEAVY:
    # E10 diagnosis (M5): how much of the spread does the filter absorb? Parameters as specified (Chan 2013), not tuned.
    from src.strategies import kalman_hedge
    logp_us = np.log(px_us)
    stats = []
    for sel in sel_us:
        w = sel['window']; nf = w.t0 - w.f0
        for spec in sel['coint']:
            fa = logp_us[spec.a].iloc[w.f0:w.t1].ffill().values
            fb = logp_us[spec.b].iloc[w.f0:w.t1].ffill().values
            betas, _, errs, qs = kalman_hedge(fa, fb, ve=spec.sd ** 2, beta0=spec.beta, mu0=spec.mu)
            static = fa[nf:] - spec.beta * fb[nf:] - spec.mu
            z = errs[nf:] / np.sqrt(qs[nf:])
            stats.append({'sd of beta in trading window': betas[nf:].std(),
                          'sd(forecast error) / sd(static spread)': errs[nf:].std() / static.std(),
                          'share of days |z| > 1': np.mean(np.abs(z) > 1), 'sd of z': z.std()})
    e10_diag = pd.DataFrame(stats).mean()
    e10_diag.to_csv('results/e10_kalman_diagnosis.csv'); display(e10_diag.round(3))

**E10 result (M5): as specified, the Kalman rule fails, and the diagnosis shows why.** (results/e10_kalman_diagnosis.csv)

- S4 has a Sharpe ratio of -0.12 (SE 0.25), only 119 trades held 3.6 days on average, and a 47% hit rate.
- With $\delta = 10^{-4}$ (Chan 2013) on log prices, $\beta_t$ moves by about 0.01 a day, so the filter absorbs most of the spread: the forecast error has only 31% of the standard deviation of the static spread.
- The observation variance is set to the formation spread variance, which is now far too large, so the standardised error has standard deviation 0.21 instead of 1 and passes the $\pm1$ entry on only 0.5% of days. The rule barely trades, and when it does the error is mostly noise.
- The principled fix is to estimate $\delta$ and the observation variance by maximum likelihood on the formation window. We leave it as future work rather than tune $\delta$ on these results.

1. **Do ADF and Engle-Granger find mean-reverting spreads?** Only fast ones: a 252-day window detects half-lives up to about 10 days (E2a), and plain ADF critical values reject 17% of independent random-walk pairs instead of 5% (E2b). Even with Engle-Granger, 8.4 of 196 US pairs pass per window, about as many as chance.
2. **How well does the OU model describe them?** The model is internally consistent (Bertram's formula matches simulation, E3; the P&L accounting reproduces theory, E4), but its estimates are not accurate: $\hat\kappa$ is biased upward by 18–96% at 252 days (E1), and realistic trading halves the theoretical return of a perfect OU pair (E4).
3. **Do OU rules beat the distance rule after costs?** In point estimates yes (US Sharpe 0.06–0.19 vs −0.05; India 0.32–0.38 vs 0.16; S1 breaks even at 22 bp vs 8 bp), and they win 63–69% of trades, but no difference is significant and the ranking shifts across sub-periods (E5–E8). The bottleneck is selecting genuinely cointegrated pairs, not the trading rule.

*Limitations:* survivorship bias (today's large caps), fixed hedge ratio within a window, daily closes only. *Future work:* Kalman hedge ratio with MLE noise, longer formation windows (E7), bias-corrected $\hat\kappa$, Leung & Li (2015) optimal stopping.